In [2]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=False)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=100,adc_last_gap=10)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(50, 50,delay3=100)
chip.add_compiler("../compiler/code/")

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
# set_good_device_file_name = "../data/good_device/cond_500_r8.npy"
# reset_good_device_file_name = "../data/good_device/cond_200_r8.npy"

In [ ]:
select = SELECT()

In [ ]:
# 并行读点，split_type=1，使用crossbar参数
crossbar = np.ones((256,256))
# v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=1,row_type=0,col_type=0)
# plot_cond(c,vmax=1000)

v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=1,row_type=0,col_type=0)
plot_cond(c,vmax=1000)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

### 1. Reset操作

In [ ]:
for i in range(1):
      select.Reset(chip=chip,need_read = np.ones((256,256),dtype=bool),write_times=11,start_v=1.5,delta_v=0.05,tg=5,threshold=200,
            reset_pulse_width=100e-6,sub_base=True,plot_cond=plot_cond,vmax=1400)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
def Set(self,chip,need_read,write_times,write_voltage,start_tg,delta_tg,threshold,set_pulse_width,sub_base=True,vmax=1000,plot_cond=None):
    cond_all = np.zeros_like(need_read,dtype=float)
    for i in range(write_times):
        print(f"write_time = {i}")
        tg = start_tg+i*delta_tg
        _,cond,_ = chip.read4(crossbar=need_read,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=sub_base,from_row=True,split_type=0,row_type=0,col_type=0)

        condition_set = (cond>threshold)&need_read
        cond_all[need_read] = cond[need_read]
        need_read = condition_set
        if plot_cond: plot_cond(cond_all,title=f"tg={tg:.2f}-needSet={np.sum(condition_set)}",vmax=vmax)

        chip.write4(crossbar=condition_set,row_index=None,col_index=None,write_voltage=write_voltage,tg=tg,pulse_width=set_pulse_width,set_device=True,split_type=0,row_type=0,col_type=0)


In [ ]:
crossbar = np.ones((256,256),dtype=bool)
Set(select,chip=chip,need_read=crossbar,write_times=10,write_voltage=5,start_tg=1.5,delta_tg=0.05,threshold=600,set_pulse_width=100e-6,sub_base=True,vmax=1000,plot_cond=plot_cond)

In [ ]:
print(np.sum(cond<200))
reset_good_device_file_name = "../data/good_device/20250613_cond_200_r8.npy"
np.save(reset_good_device_file_name,cond<200)

### 2. Set操作

In [ ]:
need_read = np.ones((256,256),dtype=bool)
# need_read[:20,120:180]=True
select.Set(chip=chip,need_read=need_read,write_times=1,write_voltage=5,start_tg=1.5,delta_tg=0.05,threshold=800,set_pulse_width=100e-6,
           sub_base=True,vmax=1000,plot_cond=plot_cond)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
print(np.sum(cond>500))
set_good_device_file_name = "../data/good_device/20250613_cond_500_r8.npy"
np.save(set_good_device_file_name,cond>500)

In [ ]:
set_good_device_file_name = "../data/good_device/20250613_cond_500_r8.npy"
reset_good_device_file_name = "../data/good_device/20250613_cond_200_r8.npy"
cond_200 = np.load(reset_good_device_file_name)
cond_500 = np.load(set_good_device_file_name)
good_cond = cond_200&cond_500
overall_good_device_file_name = "../data/good_device/20250613_cond_reset_200_set_500.npy"
print(f"# of good devices = {np.sum(good_cond)}")
np.save(overall_good_device_file_name,good_cond)